# <center> Exploring Vienna's Airbnb Market Data </center>

## Project Overview

Airbnb prices can differ across internet listings, even within the same city. These differences may be explained by/ depend on the location of the listing, property characteristics, room type, capacity, reviews, availability and host status. 

The aim of this project is to analyse and study which characteristics are most important to Airbnb listing prices in Vienna and to use the different statistical models to predict the price of a listing.

The guiding research question of this project is:
"How effectively can Airbnb listing prices in Vienna be explained and predicted, using the 11 variables defined in following section, and how does the predictive performance on the test data of Lasso Regression and Random Forest compare?"

## Data Source & Preparation Information

This project uses the listings dataset from Inside Airbnb for Vienna. (Link reference: https://insideairbnb.com/get-the-data/)

The `listings.csv` dataset contains 13,283 Airbnb listings and 90 different variables. Of those 90 variables, only 13 independent variables will be chosen, as discribed in the following section.

### Main Variables

The dependent variable is:

`price`: the nightly price of the Airbnb listing.

The selected predictor variables are:

`superhost_status`: indicates whether the host has Airbnb Superhost status. <br>
`neighbourhood`: the Vienna neighbourhood in which the listing is located. <br>
`property_type`: the type of property, such as an apartment, house or hotel. <br>
`room_type`: indicates whether the listing is an entire home, private room, shared room or hotel room. <br>
`accommodates`: the maximum number of guests that can stay in the listing. <br>
`bathrooms`: the number of bathrooms available in the listing. <br>
`bedrooms`: the number of bedrooms in the listing. <br>
`min_nights`: the minimum number of nights required for a booking. <br>
`availability_365`: the number of days during the following year on which the listing is available. <br>
`num_reviews`: the total number of reviews received by the listing. <br>
`rating`: the overall rating received by the listing.

The variable `bathrooms_text` contains the bathroom number in text form,e.g "1 bath", "1.5 baths" or "1 shared bath". It will be converted into only numbers.

`latitude` and `longitude` variables will only be used for a city heat-map showing.

All listings with missing information were removed from the dataset (dropped), and the prepared dataset which will be used will have in total 7,863 observations.

### Model Selection

For this project, Lasso Regression and Random Forest Regression are selected.

`Lasso`: a regularization method that estimates the model and selects variables at the same time. It shrinks the coefficients of less important predictors (can even set some of them to zero). <br>
This is helpful for this dataset, as the categorical variables (e,g: neighbourhood and property type) will create many dummy variables.

`Random Forest`: a more flexible tree-based method. It combines the predictions of many trees and uses a random selection of predictors when each split is made. <br>
This reduces the correlation between the trees and can lower the variance of the final predictions. It's ability to capture nonlinear relationships and interactions between the listing characteristics is also useful.

These models were chosen, in order to make a comparison between a regularized linear model and a more flexible tree-based model.

## Loading, cleaning, and preparing the data

In [7]:
# Libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

In [8]:
df = pd.read_csv('../data/listings.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13283 entries, 0 to 13282
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            13283 non-null  int64  
 1   listing_url                                   13283 non-null  str    
 2   scrape_id                                     13283 non-null  int64  
 3   last_scraped                                  13283 non-null  str    
 4   source                                        13283 non-null  str    
 5   name                                          13283 non-null  str    
 6   description                                   12917 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   13282 non-null  str    
 9   host_id                                       13283 non-null  int64  
 1

In [9]:
variables = [
    'price',
    'neighbourhood_cleansed',
    'property_type',
    'room_type',
    'accommodates',
    'amenities',
    'bedrooms',
    'bathrooms_text',
    'minimum_nights',
    'availability_365',
    'host_is_superhost',
    'number_of_reviews',
    'review_scores_rating',
    'latitude',
    'longitude'
]

airbnb = df[variables]
airbnb.head(3)

,price,neighbourhood_cleansed,property_type,room_type,accommodates,amenities,bedrooms,bathrooms_text,minimum_nights,availability_365,host_is_superhost,number_of_reviews,review_scores_rating,latitude,longitude
0,NaN,Ottakring,Entire rental unit,Entire home/apt,4,"[""Kitchen"", ""Elevator"", ""Wifi"", ""Hangers"", ""Wa...",2.0,1 bath,5.0,0,f,0,NaN,48.21136,16.32806
1,NaN,Margareten,Entire rental unit,Entire home/apt,6,"[""Kitchen"", ""Iron"", ""Elevator"", ""Wifi"", ""TV"", ...",3.0,1.5 baths,2.0,0,f,0,NaN,48.18514,16.34720
2,NaN,Leopoldstadt,Entire rental unit,Entire home/apt,4,"[""Cooking basics"", ""Stove"", ""First aid kit"", ""...",1.0,1 bath,30.0,0,f,193,4.84,48.22375,16.40081


In [10]:
airbnb = airbnb.rename(columns={
    "bathrooms_text": "bathrooms",
    "neighbourhood_cleansed": "neighbourhoods",
    "host_is_superhost": "superhost_status",
    "minimum_nights": "min_nights",
    "number_of_reviews": "num_reviews",
    "review_scores_rating": "rating"
})


# Dropping missing observations
airbnb = airbnb.dropna()

# Converting price to number
airbnb['price'] = airbnb['price'].str.replace(
    r"[$,]","", regex=True
    ).astype(float)

airbnb = airbnb.reset_index(drop= True)

# Making Superhost a binary variable
airbnb['superhost_status'] = airbnb['superhost_status'].map({
    "t": 1,
    "f": 0
})

# Fixing character problems (due to german special letters) in neighbourhood
airbnb['neighbourhoods'] = airbnb['neighbourhoods'].replace({
    "D\x9abling": "Döbling",
    "Landstra§e": "Landstraße",
    "Rudolfsheim-F\x9fnfhaus": "Rudolfsheim-Fünfhaus",
    "W\x8ahring": "Währing"
})

In [11]:
print(airbnb.isna().sum())

price               0
neighbourhoods      0
property_type       0
room_type           0
accommodates        0
amenities           0
bedrooms            0
bathrooms           0
min_nights          0
availability_365    0
superhost_status    0
num_reviews         0
rating              0
latitude            0
longitude           0
dtype: int64


In [12]:
# Converting text -> number for bathrooms
# Extracting the number  [anything without a digit ("Half-bath") -> NaN]
airbnb['bathrooms'] = pd.to_numeric(
    # \d+ -> one or more digits
    # \.? -> (if) decimal point
    # \d* -> digits of the decimal
    airbnb['bathrooms'].str.extract(r'(\d+\.?\d*)')[0], 
    errors='coerce'
)
# Fill in 0.5 for NaN
airbnb.loc[
    airbnb['bathrooms'].isna(),
    'bathrooms'
] = 0.5

In [13]:
airbnb

,price,neighbourhoods,property_type,room_type,accommodates,amenities,bedrooms,bathrooms,min_nights,availability_365,superhost_status,num_reviews,rating,latitude,longitude
0,62.33,Simmering,Entire rental unit,Entire home/apt,2,"[""Dedicated workspace"", ""Keypad"", ""Hair dryer""...",1.0,1.0,3.0,200,0,1,5.00,48.16854,16.41556
1,71.00,Simmering,Entire rental unit,Entire home/apt,5,"[""Washer"", ""Wifi"", ""Kitchen""]",1.0,1.0,1.0,217,0,4,5.00,48.17267,16.41401
2,147.40,Leopoldstadt,Entire home,Entire home/apt,4,"[""Shower gel"", ""Private patio or balcony"", ""Re...",2.0,1.0,5.0,284,0,20,4.60,48.18967,16.42396
3,32.75,Leopoldstadt,Entire rental unit,Entire home/apt,1,"[""Kitchen"", ""Iron"", ""Host greets you"", ""Wifi"",...",1.0,1.0,35.0,358,0,1,5.00,48.21946,16.37572
4,103.00,Alsergrund,Entire condo,Entire home/apt,3,"[""Freezer"", ""Bathtub"", ""Hangers"", ""Dishwasher""...",2.0,1.0,3.0,22,0,18,4.94,48.22635,16.36263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7858,43.07,Alsergrund,Entire rental unit,Entire home/apt,2,"[""Dryer"", ""Kitchen"", ""Iron"", ""Dishes and silve...",1.0,1.0,60.0,214,0,8,5.00,48.21535,16.34661
7859,50.25,Margareten,Entire rental unit,Entire home/apt,2,"[""Stainless steel oven"", ""Blender"", ""Luggage d...",1.0,1.0,1.0,81,1,33,4.82,48.18636,16.36602
7860,58.52,Mariahilf,Entire condo,Entire home/apt,2,"[""Cooking basics"", ""Stove"", ""First aid kit"", ""...",1.0,1.0,30.0,59,1,191,4.82,48.19050,16.34743
7861,18.62,Leopoldstadt,Entire rental unit,Entire home/apt,4,"[""Blender"", ""Heating"", ""Cooking basics"", ""Firs...",2.0,1.0,90.0,337,0,22,4.77,48.21765,16.37582
